In [2]:
import pandas as pd
import requests
from jsonschema import validate, ValidationError
from accessibility import check_endpoint
from summarize import * #json_keys, data_frame

### Interoperability Assessment of EuroArgo JSON api (Floats data)

**Endpoint documentation**

API documentation (swagger): https://dataselection.euro-argo.eu/swagger-ui/index.html

### Endpoint

In [3]:
base = "https://dataselection.euro-argo.eu"

##### Table Of Content

- [Exploring the JSON API](#exploring-the-json-api)
- [Technical interoperability](#technical-interoperability)
    - [Standards compliance](#standards-compliance) 
    - [Interface consistency](#interface-consistency) 
    - [Versioning & Backward compatibility](#versioning-and-backwards-compatibility) 

- [Semantic interoperability](#semantic-interoperability)
    - [Documentation](#documentation) 
    - [Metadata](#metadata)
    - [Shared Vocabulary & Ontologies](#shared-vocabularies-and-ontologies)
    - [Contextual Meaning](#contextual-meaning) 
- ([note on Data Granularity](#note-on-data-granularity))

### Exploring the JSON API

Exploration of the API was done through the swagger documentation and for GET requests also below in this notebook:

**Basins** (get basins names as a tree)

In [5]:
basins_url = "https://dataselection.euro-argo.eu/api/basins-tree"

basin_md = requests.get(basins_url).json()

summary = json_keys(basin_md)

basin_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
basin_md_summ.to_csv("files_euroargo_jsonapi/ARGO_JSONAPI_floatsdata_basins.csv", index=False)
basin_md_summ

,Property,Count,Types,Example
0,id,11,int,23617
1,name,11,str,ARCTIC OCEAN
2,subBasins,11,list,"[{'id': 24027, 'name': 'KARA SEA'}, {'id': 240..."
3,subBasins.id,236,int,26579
4,subBasins.name,236,str,FRAM STRAIT


This endpoint returns a list of 11 ocean basins and 236 sub-basins. The data model can be visualized as:  
![image.png](images/ARGO_JSONAPI_oceanbasins.drawio.png)

**Cycle by cycle-id** (get a cycle metadata by cycleId)

In [6]:
cycle_url = "https://dataselection.euro-argo.eu/api/find-by-id/760094"

cycle_md = requests.get(cycle_url).json()

summary = json_keys(cycle_md)

cycle_md_summ= pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
cycle_md_summ.to_csv("files_euroargo_jsonapi/ARGO_JSONAPI_floatsdata_cycle.csv", index=False)
cycle_md_summ

,Property,Count,Types,Example
0,id,1,int,760094
1,cvNumber,1,int,1
2,startDate,1,str,2008-03-23T02:00:54.000+0000
3,endDate,1,str,2008-03-23T02:00:54.000+0000
4,coordinate,1,dict,"{'lat': -28.184, 'lon': -22.997}"
5,coordinate.lat,1,float,-28.184
6,coordinate.lon,1,float,-22.997
7,globalGeoShapeField,1,dict,"{'x': -22.997, 'y': -28.184, 'type': 'Point', ..."
8,globalGeoShapeField.x,1,float,-22.997
9,globalGeoShapeField.y,1,float,-28.184


The endpoint returns metadata associated with a specific cycle Id. The data model can be visualized as follows:  
![image.png](images/ARGO_JSONAPI_cycle.drawio.png)

**Cycle by platformID and cvNumber** (get a cycle metadata by platformId and cvNumber)

In [7]:
cycle2_url = "https://dataselection.euro-argo.eu/api/find-by-platformid/3900675/cvnumber/1"

cycle2_md = requests.get(cycle2_url).json()

summary = json_keys(cycle2_md)

cycle2_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
cycle2_md_summ.to_csv("files_euroargo_jsonapi/ARGO_JSONAPI_floatsdata_cycle.csv", index=False)
cycle2_md_summ
# returns same cycle metadata as 'Cycle by cycle-id'

,Property,Count,Types,Example
0,id,1,int,760094
1,cvNumber,1,int,1
2,startDate,1,str,2008-03-23T02:00:54.000+0000
3,endDate,1,str,2008-03-23T02:00:54.000+0000
4,coordinate,1,dict,"{'lat': -28.184, 'lon': -22.997}"
5,coordinate.lat,1,float,-28.184
6,coordinate.lon,1,float,-22.997
7,globalGeoShapeField,1,dict,"{'x': -22.997, 'y': -28.184, 'type': 'Point', ..."
8,globalGeoShapeField.x,1,float,-22.997
9,globalGeoShapeField.y,1,float,-28.184


**Trajectory by platformID** (get a cycle trajectory by platformId)

In [8]:
trajectory_url = "https://dataselection.euro-argo.eu/api/trajectory/3900675"

traj_md = requests.get(trajectory_url).json()

summary = json_keys(traj_md)

traj_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
traj_md_summ.to_csv("files_euroargo_jsonapi/ARGO_JSONAPI_floatsdata_trajectory.csv", index=False)
traj_md_summ


,Property,Count,Types,Example
0,id,116,int,1199770
1,cvNumber,116,int,60
2,coordinate,116,dict,"{'lat': -27.449, 'lon': -33.858}"
3,coordinate.lat,116,float,-27.449
4,coordinate.lon,116,float,-33.858
5,level,116,int,0


The endpoint returns metadata associated with a specific trajectory based on a cycle Id. The data model can be visualized as follows:  
![image.png](images/ARGO_JSONAPI_traj.drawio.png)

API returns:
  - 19045 Floats:
    - described by 39 properties (many more when taking nested properties into account (see diagram for overview))

  - 11 Basins:
    - described by 3 properties
    - 236 sub-basins

  - 900+ Cycles:
    - described by 30 properties (51 taking nested properties into account)
    - no exact number on total number of cycles

### Technical Interoperability

#### Standards compliance 
*Involves checking compliance to HTTP standard, JSON schema, JSON API spec*

In [4]:
# various endpoints
urls = [
    'https://dataselection.euro-argo.eu/api',                                           #base
    'https://dataselection.euro-argo.eu/api/basins-tree',                               #example endpoint from documentation
    'https://dataselection.euro-argo.eu/api/find-by-id/760094',                         #example endpoint from documentation, with specified input parameter 
    'https://dataselection.euro-argo.eu/api/find-by-id',                                #faulty endpoint - missing input parameter 'station ID'
    'https://dataselection.euro-argo.eu/api/find-by-platformid/3900675/cvnumber/1',     #example endpoint from documentation, with multiple specified input parameters 
    'https://dataselection.euro-argo.eu/api/find-by-platformid/3900675',                #faulty endpoint
    'https://dataselection.euro-argo.eu/api/find-by-platformid/3900675/cvnumber/',      #faulty endpoint
    'https://dataselection.euro-argo.eu/api/trajectory/3900675'                         #example endpoint from documentation, with specified input parameter 
]

# headers of interest
important_headers = [
    "Content-Type",
    "Content-Length",
    "Content-Encoding",
    "Transfer-Encoding",
    "Vary",
    "Accept",
    "Accept-Encoding",
    "Accept-Language",
    "Server",
    "Date",
    "Strict-Transport-Security"
]

for url in urls:
    print("="*100)
    print(f"🔗 URL: {url}")
    try:
        r = requests.get(url, timeout=10)
        print(f"Status code: {r.status_code}\n")

        # filter only important headers
        headers_filtered = {k: v for k, v in r.headers.items() if k in important_headers}
        
        print("📌 Relevant headers:")
        if headers_filtered:
            for k, v in headers_filtered.items():
                print(f"   {k}: {v}")
        else:
            print("   (none of the selected headers present)")
        
        # try parsing response
        print("\n📌 Response preview:")
        try:
            data = r.json()
            if isinstance(data, dict):
                print("   JSON object with keys:", list(data.keys()))
            elif isinstance(data, list):
                print(f"   JSON list with {len(data)} items")
            else:
                print("   JSON response (other type)")
        except ValueError:
            print("   Text response (first 300 chars):")
            print("   " + r.text[:300].replace("\n", " ") + " ...")
    
    except requests.exceptions.RequestException as e:
        print(f"❌ Request failed: {e}")

🔗 URL: https://dataselection.euro-argo.eu/api
Status code: 200

📌 Relevant headers:
   Date: Wed, 27 Aug 2025 11:53:58 GMT
   Server: Apache
   Content-Type: text/html
   Vary: Accept-Encoding
   Content-Encoding: gzip
   Transfer-Encoding: chunked

📌 Response preview:
   Text response (first 300 chars):
   <!doctype html> <html lang="en" data-beasties-container> <head>   <meta charset="utf-8">   <title>Euro Argo Data Selection</title>   <base href="/">   <meta name="viewport" content="width=device-width, initial-scale=1">   <meta name="robots" content="index,nofollow">   <link rel="icon" type="image/x ...
🔗 URL: https://dataselection.euro-argo.eu/api/basins-tree
Status code: 200

📌 Relevant headers:
   Date: Wed, 27 Aug 2025 11:53:58 GMT
   Server: Apache
   Content-Type: application/json
   Transfer-Encoding: chunked

📌 Response preview:
   JSON list with 11 items
🔗 URL: https://dataselection.euro-argo.eu/api/find-by-id/760094
Status code: 200

📌 Relevant headers:
   Date: Wed, 27 Au

In [ ]:
# Cycle metadata schema (from api documentation)
schema = {
  "highlightFields": {
    "additionalProp1": [
      "string"
    ],
    "additionalProp2": [
      "string"
    ],
    "additionalProp3": [
      "string"
    ]
  },
  "id": 0,
  "cvNumber": 0,
  "startDate": "2025-08-18T11:54:25.402Z",
  "endDate": "2025-08-18T11:54:25.402Z",
  "coordinate": {
    "lat": 0,
    "lon": 0
  },
  "globalGeoShapeField": {
    "type": "string",
    "coordinates": {}
  },
  "platformCode": "string",
  "positionQc": "string",
  "dateQc": "string",
  "cycleQcState": "string",
  "grounded": "true",
  "pmax": 0,
  "surfacePressure": 0,
  "surfaceTemperature": 0,
  "surfaceSalinity": 0,
  "bottomPressure": 0,
  "bottomTemperature": 0,
  "bottomSalinity": 0,
  "stations": [
    {
      "stationId": 0,
      "coordinate": {
        "lat": 0,
        "lon": 0
      },
      "cvNumber": 0,
      "positionQc": "string",
      "dateQc": "string",
      "comment": "string",
      "type": "string",
      "direction": "string",
      "date": "2025-08-18T11:54:25.402Z",
      "pmax": 0,
      "eol": "2025-08-18T11:54:25.402Z",
      "surface": "2025-08-18T11:54:25.402Z",
      "level": 0,
      "primary": "true",
      "syntheticSampling": "true",
      "physicalParameters": [
        0
      ]
    }
  ],
  "stationsId": [
    0
  ],
  "basins": [
    "string"
  ],
  "parameters": [
    "string"
  ],
  "networkCodes": [
    "string"
  ],
  "groupCodes": [
    "string"
  ],
  "deploymentDate": "2025-08-18T11:54:25.402Z",
  "deploymentYear": "string",
  "countryLabel": "string",
  "institutionLabel": "string",
  "telecomCode": "string",
  "dacCode": "string",
  "platformTypeCode": "string",
  "getpI": "string",
  "cruiseCode": "string",
  "statusCode": "string",
  "sensors": [
    "string"
  ],
  "dataMode": "string",
  "lastUpdateDate": "2025-08-18T11:54:25.402Z",
  "level": 0,
  "physicalParameters": [
    {
      "code": 0,
      "labelWithCode": "string"
    }
  ]
}

# Step 2: Fetch JSON from the API (replace with the right endpoint)
url = "https://dataselection.euro-argo.eu/api/find-by-id/760094"   # <-- update endpoint
response = requests.get(url)
data = response.json()

# Step 3: Validate the JSON response against the schema
try:
    validate(instance=data, schema=schema)
    print("✅ JSON conforms to schema")
except ValidationError as e:
    print("❌ JSON validation error:", e.message)

✅ JSON conforms to schema


In [ ]:
# Basin-tree metadata schema (from api documentation)
schema = {
    "id": 0,
    "name": "string",
    "subBasins": [
      "string"
    ],
    "extent": {
      "coordinates": [
        {
          "coordinates": [
            {
              "type": "string",
              "coordinates": [
                {
                  "x": 0,
                  "y": 0
                }
              ]
            }
          ],
          "type": "string"
        }
      ],
      "type": "string"
    }
  }

# Step 2: Fetch JSON from the API (replace with the right endpoint)
url = "https://dataselection.euro-argo.eu/api/basins-tree"   # <-- update endpoint
response = requests.get(url)
data = response.json()

# Step 3: Validate the JSON response against the schema
try:
    validate(instance=data, schema=schema)
    print("✅ JSON conforms to schema")
except ValidationError as e:
    print("❌ JSON validation error:", e.message)

✅ JSON conforms to schema


#### Interface consistency
*Involves checking naming conventions, field types and data formats, error messages, cross-platform support (language neutral data representation, no additional data transformations needed when consuming the data)*

In [ ]:
import requests
import json

def get_openapi_spec(base_url):
    for path in ["/v2/api-docs", "/v3/api-docs", "/openapi.json", "/swagger.json"]:
        url = base_url.rstrip("/") + path
        r = requests.get(url)
        if r.ok and "json" in r.headers.get("content-type", ""):
            return r.json()
    return None

# Example: Euro-Argo Fleet Monitoring API
spec = get_openapi_spec(base)
print(json.dumps(spec, indent=2))

endpoint = "/api/platforms"   # choose the endpoint you want to inspect
if spec and endpoint in spec["paths"]:
    details = spec["paths"][endpoint]
    print(json.dumps(details, indent=2))
else:
    print(f"Endpoint {endpoint} not found in spec")

{'openapi': '3.0.1', 'info': {'title': 'EADS APi', 'description': 'EADS APi', 'version': '0.1.16'}, 'servers': [{'url': 'https://dataselection.euro-argo.eu', 'description': 'Generated server url'}], 'paths': {'/api/maintenance': {'get': {'tags': ['maintenance-controller'], 'operationId': 'clearMaintenance', 'parameters': [{'name': 'api-key', 'in': 'header', 'required': True, 'schema': {'type': 'string'}}], 'responses': {'200': {'description': 'OK'}}}, 'post': {'tags': ['maintenance-controller'], 'operationId': 'setMaintenance', 'parameters': [{'name': 'api-key', 'in': 'header', 'required': True, 'schema': {'type': 'string'}}], 'requestBody': {'content': {'application/json': {'schema': {'$ref': '#/components/schemas/MaintenanceDto'}}}, 'required': True}, 'responses': {'200': {'description': 'OK'}}}}, '/api/full-search-response': {'post': {'tags': ['search-controller'], 'summary': 'Get Cycles cluster groups, total count and facet criteria list for a searchEntry', 'operationId': 'getSearc

#### Versioning & Backward compatibility
*Involves checking whether the API has a clear versioning strategy and maintains support for older clients when changes to the schema occur*

In [5]:
import requests

# Try typical OpenAPI/Swagger spec URLs
urls = [
    "https://dataselection.euro-argo.eu/v3/api-docs",
    "https://dataselection.euro-argo.eu/v2/api-docs",
    "https://dataselection.euro-argo.eu/openapi.json",
]

for url in urls:
    try:
        r = requests.get(url, timeout=10)
        if r.ok and r.headers.get("content-type", "").startswith("application/json"):
            data = r.json()
            version = data.get("info", {}).get("version")
            print(f"{url} --> version: {version}")
    except Exception as e:
        print(f"{url} --> failed ({e})")

https://dataselection.euro-argo.eu/v3/api-docs --> version: 0.1.16


Findings:

**Standards compliance**  
Fidnings on compliance to HTTP protocol:
- compliant to HTTP protocol
- only correct endpoints listed in swagger api documentation results in valid json response
- other endpoints, either without specified input parameters or with invalid input parameters, do not return the correct application/json content-type ~ improvements could be made
- JSON schemas are defined and available through the documentation and responses from tested endpoints adhere to schemas ~ conformance to JSON schema 

Sidenote: 
- not a RESTful api ~ base endpoints, none existing endpoints return wrong content-type in header

**Interface consistency**  
- use of camelCase naming convention
- no correct error message for wrongly formatted endpoints 


**Versioning & Backward compatibility**  
backward compatibility between api version is an important aspect of technical interoperability.
However, since we have only access to the latest version (V3) and other versions return a 404, we cannot verify this.


Overall, the data is findable, machine accessible and, given knowledge on API structure (which is available through the Swagger API documentation), easily navigatable/usable. 

### Semantic Interoperability

#### Documentation
*Involves checking availablity and content of swagger documentation (e.g. definitions include descriptions not just types)*

API documentation available at https://dataselection.euro-argo.eu/swagger-ui/index.html  


The Swagger API documentation clearly lists the available endpoints for a specific HTTP method.  
The listed endpoints can be dynamically explored and tested (with or without input parameters).  

The schemas definitions only include datatypes. Descriptions are not given, making the the meaning of definitions ambiguous to non-domain experts.  

![example-schema-definition](./images/EuroArgo_JsonAPI_exampleschemadefinition.png)

Also in the dynamical exploration of the endpoints, the definitions of input parameters are not given, nor any example values (this is understandable in post requests not in get requests); only datatypes are given.  
Some endpoints require knowledge on wmo-codes, platform-codes, ...

![example-input-parameters](./images/EuroArgo_JsonAPI_exampleinputparameters.png)


#### Metadata 
*Involves checking use of self-describing/unambiguous terms, measurements of units defined, ...*

Example response body of get request for a cycle trajectory by platformId indicates:

- Terms appear to be self-describing (e.g. `lat` and `lon` ~ latitude and longitude),  
however due to lack descriptions in the schema definitions, terms are still ambiguous (e.g. `cycleQcState` -> the quality control state of a cycle; but what is a cycle?) 

- Measurements of units are not defined  
(e.g. surfacePressure, surfaceTemperature, surfaceSalinity values are represented by integer, the unit of measurement is not captured here, nor in antoher key in the response body)  

![example-response-body](./images/EuroArgo_JsonAPI_exampleresponsebody.png)

#### Shared Vocabulary & Ontologies
*Involves checking use of terms from standard vocabularies (ISO, schema.org, ...) and, if applicable, alignment with domain ontologies*

- Dates formats are following ISO 8601 standard.

- Limited/no explicit adherence to other standards (like schema.org, DCAT) or domain standards. This is not mentioned in documentation.   
(though this is not an API that returns JSON-LD, part of used terms do seems to align with terms from schema.org).  

- Limited / no use of persistent identifiers (URIs) - e.g. wmo-codes/platform-codes, types, ... could be represented by URLs from existing standards. 

#### Contextual Meaning
*Involves checking context terms (e.g. `"status":"active"`) and enumerations are clearly defined so their meaning is consistent across different systems*

- no notes

**Note on data granularity**

to include!